## Example 1: Basic Event Emitter Usage

In [ ]:
from pyee import EventEmitter

# Example 1: Basic event emitter operations
print("=== Basic EventEmitter Usage ===")

# Create an event emitter instance
emitter = EventEmitter()

# Define event handlers
def user_login_handler(username):
    print(f"User logged in: {username}")

def security_log_handler(username):
    print(f"Security: Login attempt for {username}")

# Register event listeners
emitter.on('user_login', user_login_handler)
emitter.on('user_login', security_log_handler)

print(f"Registered listeners: {len(emitter.listeners('user_login'))}")
print(f"Event names: {emitter.event_names()}")

# Emit events
print("\nEmitting login events:")
emitter.emit('user_login', 'alice')
emitter.emit('user_login', 'bob')

# Test emit return value
success = emitter.emit('user_login', 'charlie')
print(f"Event emission successful: {success}")

# Test non-existent event
no_listeners = emitter.emit('nonexistent_event', 'data')
print(f"Non-existent event emission: {no_listeners}")

print("✓ Basic event emitter functionality demonstrated")

## Example 2: One-time Events and Listener Management

In [ ]:
from pyee import EventEmitter

# Example 2: One-time events and listener management
print("=== One-time Events and Listener Management ===")

emitter = EventEmitter()

# Define handlers
def startup_handler(config):
    print(f"Application started with config: {config}")

def one_time_init_handler(data):
    print(f"One-time initialization: {data}")

def persistent_handler(data):
    print(f"Persistent handler: {data}")

# Register one-time listener
emitter.once('app_startup', one_time_init_handler)

# Register persistent listener
emitter.on('app_startup', startup_handler)
emitter.on('app_startup', persistent_handler)

print(f"Initial listeners for 'app_startup': {len(emitter.listeners('app_startup'))}")

# First emission - all handlers should run
print("\nFirst app startup:")
emitter.emit('app_startup', {'debug': True, 'port': 8080})

print(f"After first emit - listeners: {len(emitter.listeners('app_startup'))}")

# Second emission - only persistent handlers run
print("\nSecond app startup:")
emitter.emit('app_startup', {'debug': False, 'port': 3000})

# Demonstrate listener removal
emitter.remove_listener('app_startup', persistent_handler)
print(f"After removing persistent handler: {len(emitter.listeners('app_startup'))}")

# Remove all listeners
emitter.remove_all_listeners('app_startup')
print(f"After remove_all_listeners: {len(emitter.listeners('app_startup'))}")

print("✓ One-time events and listener management demonstrated")

## Example 3: Asynchronous Event Handling

In [ ]:
from pyee.asyncio import AsyncIOEventEmitter
import asyncio

# Example 3: Asynchronous event handling
print("=== Asynchronous Event Handling ===")

async def demo_async_events():
    # Create AsyncIO-compatible event emitter
    async_emitter = AsyncIOEventEmitter()
    
    # Define async event handlers
    async def async_file_processor(filename):
        print(f"Started processing file: {filename}")
        await asyncio.sleep(0.1)  # Simulate async file processing
        print(f"Completed processing file: {filename}")
    
    async def async_notification_sender(filename):
        await asyncio.sleep(0.05)  # Simulate async network call
        print(f"Notification sent for: {filename}")
    
    # Define sync handler for comparison
    def sync_logger(filename):
        print(f"Log entry: File {filename} queued for processing")
    
    # Register mixed sync and async handlers
    async_emitter.on('file_upload', async_file_processor)
    async_emitter.on('file_upload', async_notification_sender)
    async_emitter.on('file_upload', sync_logger)
    
    print(f"Registered handlers: {len(async_emitter.listeners('file_upload'))}")
    
    # Emit events and demonstrate async handling
    print("\nEmitting file upload events:")
    async_emitter.emit('file_upload', 'document1.pdf')
    async_emitter.emit('file_upload', 'image2.jpg')
    
    # Allow async handlers to complete
    await asyncio.sleep(0.2)
    
    return async_emitter

# Run async demo
async_emitter = await demo_async_events()
print(f"Async emitter event names: {async_emitter.event_names()}")
print("✓ Asynchronous event handling demonstrated")

## Example 4: Event System with Multiple Arguments and Error Handling

In [ ]:
from pyee import EventEmitter

# Example 4: Multiple arguments and error handling
print("=== Multiple Arguments and Error Handling ===")

emitter = EventEmitter()

# Handler with multiple arguments and keyword arguments
def order_processor(*args, **kwargs):
    order_id, customer_id, amount = args[:3]
    priority = kwargs.get('priority', 'normal')
    print(f"Processing order {order_id} for customer {customer_id}")
    print(f"  Amount: ${amount}, Priority: {priority}")

# Error-prone handler
def payment_handler(order_id, customer_id, amount, **kwargs):
    if amount > 10000:
        print(f"⚠️  High-value transaction flagged: ${amount}")
    else:
        print(f"Payment processed: ${amount} for order {order_id}")

# Safe error handling wrapper: absorbs ONLY the rejection the risky handler raises on purpose
rejected_orders = []
def safe_handler(handler_name):
    def wrapper(*args, **kwargs):
        try:
            print(f"Executing {handler_name} handler...")
            # Simulate handler logic
            if handler_name == "risky":
                if len(args) > 0 and args[2] > 5000:  # amount check
                    raise ValueError("Transaction requires manual approval")
            print(f"✓ {handler_name} handler completed successfully")
        except ValueError as e:
            assert str(e) == "Transaction requires manual approval", e
            rejected_orders.append(args[0])
            print(f"✗ {handler_name} handler rejected {args[0]} ({e})")
    return wrapper

# Register handlers
emitter.on('order_created', order_processor)
emitter.on('order_created', payment_handler)
emitter.on('order_created', safe_handler('risky'))

print(f"Registered order handlers: {len(emitter.listeners('order_created'))}")

# Emit events with multiple arguments
print("\nProcessing various orders:")
emitter.emit('order_created', 'ORD-001', 'CUST-123', 150.00, priority='high')
emitter.emit('order_created', 'ORD-002', 'CUST-456', 15000.00, priority='urgent')
emitter.emit('order_created', 'ORD-003', 'CUST-789', 75.50)
assert rejected_orders == ['ORD-002'], rejected_orders  # only the 15000.00 order is over 5000

# Test listener checking
print(f"\nEvent system status:")
print(f"Listening to 'order_created': {emitter.listens_to('order_created') is not None}")
print(f"Total active events: {len(emitter.event_names())}")

print("✓ Multiple arguments and error handling demonstrated")

## Example 5: Real-world Application - Message Bus System

In [ ]:
from pyee import EventEmitter
from datetime import datetime
import time

# Example 5: Real-world message bus system
print("=== Message Bus System Implementation ===")

class MessageBus(EventEmitter):
    """A message bus system using PyEE for microservices communication"""
    
    def __init__(self):
        super().__init__()
        self.message_count = 0
        self.start_time = time.time()
        
    def publish(self, topic, message, source_service=None):
        """Publish a message to a topic"""
        self.message_count += 1
        timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        
        message_data = {
            'id': self.message_count,
            'timestamp': timestamp,
            'source': source_service or 'unknown',
            'payload': message
        }
        
        print(f"📤 Publishing to '{topic}': {message}")
        self.emit(topic, message_data)
        return message_data['id']
    
    def get_stats(self):
        """Get bus statistics"""
        uptime = time.time() - self.start_time
        return {
            'total_messages': self.message_count,
            'uptime_seconds': round(uptime, 2),
            'active_topics': len(self.event_names()),
            'topics': list(self.event_names())
        }

# Create message bus instance
bus = MessageBus()

# Define service handlers
def user_service_handler(message):
    user_id = message['payload'].get('user_id')
    action = message['payload'].get('action')
    print(f"👤 User Service: Processing {action} for user {user_id}")

def email_service_handler(message):
    recipient = message['payload'].get('email')
    print(f"📧 Email Service: Sending notification to {recipient}")

def audit_service_handler(message):
    print(f"📋 Audit Service: Logged event from {message['source']} at {message['timestamp']}")

# Register services as subscribers
bus.on('user.created', user_service_handler)
bus.on('user.created', email_service_handler)
bus.on('user.created', audit_service_handler)

bus.on('user.login', audit_service_handler)
bus.on('order.placed', audit_service_handler)

print("Registered message bus subscribers")

# Simulate microservices communication
print("\n🚀 Simulating microservice events:")

# User registration event
bus.publish('user.created', {
    'user_id': 12345,
    'action': 'registration',
    'email': 'alice@example.com'
}, 'api-gateway')

# User login event  
bus.publish('user.login', {
    'user_id': 12345,
    'action': 'login',
    'ip_address': '192.168.1.100'
}, 'auth-service')

# Order event
bus.publish('order.placed', {
    'order_id': 'ORD-789',
    'user_id': 12345,
    'amount': 99.99
}, 'order-service')

# Display message bus statistics
print(f"\n📊 Message Bus Statistics:")
stats = bus.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

print("✓ Real-world message bus system demonstrated")